# Qalqan Audio Trigger — Google Colab

Подготовка данных (S11b) и обучение модели A (S12). Код — из архива `qalqan_<коммит>.zip` на Google Диске; модели и данные скачиваются из первоисточников со сверкой хешей.

Для обучения включите GPU: *Среда выполнения → Сменить среду выполнения → T4 GPU*.

## 1. Google Диск и код проекта
`DRIVE_DIR` — папка на Диске, куда загружен архив.

In [ ]:
import glob
import os

from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/qalqan'
bundle = sorted(glob.glob(f'{DRIVE_DIR}/qalqan_*.zip'))[-1]
print('архив:', bundle)
!rm -rf /content/qalqan && unzip -q "$bundle" -d /content
os.chdir('/content/qalqan')
!ls

## 2. Установка и ассеты (~10–15 мин; FSD50K — 6,3 ГБ)

In [ ]:
!bash scripts/colab_setup.sh kws

## 3. Проверка: тесты проекта

In [ ]:
!uv run pytest -q 2>&1 | tail -3

## 4. S11b: синтез фраз (Piper + проверка Vosk) и сборка набора

In [ ]:
!uv run python -m qat.data.kws_dataset synth

In [ ]:
!uv run python -m qat.data.kws_dataset build

## 5. Сохранить набор на Диск
Случайный перезапуск Colab стирает `/content` — результат лучше сразу копировать.

In [ ]:
!tar -czf kws_dataset.tar.gz data/processed/kws
!cp kws_dataset.tar.gz "$DRIVE_DIR"/ && ls -lh "$DRIVE_DIR"

## 6. S12: обучение модели A
Ячейка появится вместе с сервисом S12.